In [3]:
import pandas as pd 
import numpy as np

In [4]:
df = pd.read_parquet('../merged_data/merged_churn_data.parquet')

In [5]:
df.columns

Index(['customer_id', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count'],
      dtype='str')

In [6]:
#churn rate
churn_rate = df['churn_flag'].mean()*100
print(f"churn rate {churn_rate.round(2)} %")


churn rate 28.57 %


In [7]:
# retentiion rate
retention_rate = 100 - churn_rate
print(f"retention rate {retention_rate.round(2)} %")

retention rate 71.43 %


In [8]:
# churn by plane type
churn_by_plan = df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name='churn_rate_pct')
print(churn_by_plan)

  plan_type  churn_rate_pct
0     Basic           60.00
1   Premium           14.29
2  Standard           22.22


In [9]:
# Churn by state + sum(revenue) & count of users
churn_by_state = df.groupby('state').agg(
    churn_rate=('churn_flag', 'mean'),
    total_revenue=('monthly_charges', 'sum'),
    user_count=('customer_id', 'count')
).reset_index()

# round churn_rate to percentage
churn_by_state['churn_rate'] = churn_by_state['churn_rate'].mul(100).round(2)
print(churn_by_state)

           state  churn_rate  total_revenue  user_count
0          Delhi       25.00          52.96           4
1      Karnataka      100.00          20.98           2
2      Kathmandu        0.00          20.98           2
3    Maharashtra        0.00          50.97           3
4      Meghalaya       66.67          42.97           3
5       Nagaland        0.00          22.99           1
6      Rajasthan        0.00          36.98           2
7      Telangana       50.00          30.98           2
8  Uttar Pradesh        0.00         115.98           2


In [10]:
# Churn by subscription type + sum(revenue) & count of users
churn_by_subs = df.groupby('subscription_type').agg(
    churn_rate = ('churn_flag' , 'mean'),
    total_revenue = ('monthly_charges' , 'sum'),
    user_count = ('customer_id' , 'count')
).reset_index()

churn_by_subs['churn_rate'] = churn_by_subs['churn_rate'].mul(100).round(2)

print(churn_by_subs)

  subscription_type  churn_rate  total_revenue  user_count
0           Organic        0.00         145.91           9
1              Paid       16.67         174.94           6
2          Refferal       83.33          74.94           6


In [11]:
#ARPU - Avg Revenue per user
arpu = df['monthly_charges'].mean()
print(arpu.round(2))

18.85


In [12]:
# Average customer tenure

today = pd.Timestamp.today()

df['tenure_days'] = np.where(
        df['cancellation_date'].notna(),
            (df['cancellation_date'] - df['subscription_start_date']).dt.days,
                    (today - df['subscription_start_date']).dt.days)

avg_tenure = df['tenure_days'].mean()
print('Average Tenure days =',round(avg_tenure),0)


Average Tenure days = 1542 0


In [13]:
# calculate customer age 

today = pd.Timestamp.today()

df['dob'] = pd.to_datetime(df['dob'], errors='coerce')

df['age'] = today.year - df['dob'].dt.year

df['age'] -= (
    (df['dob'].dt.month > today.month) |
    ((df['dob'].dt.month == today.month) & 
     (df['dob'].dt.day > today.day))
).astype(int)

avg_age = df['age'].mean()
print('Average Age of Customer =',(avg_age).round(0))

Average Age of Customer = 37.0


In [14]:
# 7. Revenue at risk - revenue lost from churned users
revenue_at_risk = df.loc[df['churn_flag'] == 1 , 'monthly_charges'].sum()
print("Revenue at risk (Rs'K') =",revenue_at_risk)


Revenue at risk (Rs'K') = 73.94


In [15]:
# 8. Esclation Rate
esclation_rate = (df['escalations']  == 'Y').mean()*100
print('Esclation Rate =',esclation_rate.round(2),'%')


Esclation Rate = 19.05 %


In [16]:
# Avg complaint per user

avg_complaint = df['complaint_count'].sum() / df['customer_id'].nunique()
print('Average complaint count =',avg_complaint.round(2))

Average complaint count = 0.43


In [17]:
# 10. Correlation Esclation vs Churn
df['escalations'] = np.where(df['escalations'] == 'Y', 1, 0) # encoding string to int type
corr_df = df[['escalations', 'churn_flag']].dropna()

correlation = corr_df['escalations'].corr(df['churn_flag'])
print("Correlation between esclation vs churn is = ", round(correlation,2))

Correlation between esclation vs churn is =  0.77


In [18]:
# 11. Create a column using existing col - Churn risk
conditions = [
        (df['churn_score'] < 50),
        (df['churn_score'] >= 50) & (df['churn_score'] < 70),
        (df['churn_score'] >= 70)
]

choices = ['low', 'med', 'high']

df['churn_risk'] = np.select(conditions, choices, default='unkown')

In [19]:
df.head(5)

,customer_id,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,state,gender,dob,complaint_date,escalations,csat_score,complaint_count,tenure_days,age,churn_risk
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,NaN,13.99,627,...,Maharashtra,Male,1982-04-12,NaT,0,NaN,NaN,2014.0,44,low
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,Karnataka,Male,1995-11-23,2024-08-28,1,10.0,2.0,1501.0,30,high
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,NaN,6.99,210,...,Delhi,Female,1978-02-15,NaT,0,NaN,NaN,1399.0,48,low
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,NaN,22.99,1725,...,Nagaland,Male,2001-08-30,NaT,0,NaN,NaN,2689.0,25,low
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,...,Delhi,Female,1990-05-05,2024-01-20,1,20.0,1.0,419.0,36,high


In [20]:
# creating a new parquet file for visualization purpose
df.to_parquet('../cleaned_data/df_visual_data.parquet', index=False)

In [21]:
# creating a new csv file for power BI visualization purpose
df.to_csv('../cleaned_data/churn_data.csv', index=False)